# Merinos Halı Sanayi ve Ticaret A.Ş. — Endüstriyel Yapay Zeka Stajı
## Gün 29: Açık Kaynaklı Minimal SLM Seçimi, Özel NLP Kütüphanesi ile Nano-LLM Mimarisi & Donanım Kuantalama Profillemesi

**Aşama:** Faz 5: Fine-Tuning & LLM Özelleştirme (Day 29–35)  
**Yazar:** Seydi Eryılmaz (@seydivakkas)  
**Telif Hakkı:** (c) 2026 Seydi Eryılmaz. Tüm Hakları Saklıdır (All Rights Reserved).

---

### 🎯 Gün 29 Hedefleri ve Endüstriyel Bağlam:
1. **Küçük Dil Modelleri (SLM) Seçimi:** Merinos Gaziantep 4. OSB dokuma fabrikasındaki uç cihazlar (RTX 4060 8GB, RTX 3060 12GB) ve on-premise A10G sunucular için en verimli açık kaynaklı SLM adaylarını (Qwen2.5-0.5B/1.5B, SmolLM2-360M, TinyLlama-1.1B, Llama-3.2-1B) belirlemek.
2. **Sıfırdan Özel Nano-LLM Motoru:** Halı dokuma terimlerine duyarlı altkelime tokenizer'ı (`MerinosBPETokenizer`), Döner Konum Gömme (`RoPE`), `RMSNorm`, `SwiGLU` FFN ve `CausalSelfAttention` (GQA) içeren modern bir Causal LM motoru kodlamak.
3. **Modern Mimari Eklentileri (MoE, Shared Expert, KV Cache, Soft-Capping):** Sparse Mixture of Experts, DeepSeek tarzı shared expert, hızlı $O(1)$ KV önbellekleme ve Gemma-2 logit soft-capping entegrasyonu.
4. **PyTorch vs. TensorFlow Karşılaştırması:** Tensör manipülasyonu, autograd vs. GradientTape, `torch.einsum` ve optimizasyon mekaniklerinin matematiksel eşdeğerlik haritası.
5. **Donanım Profilleme & Kuantalama:** Ağırlık ve KV önbelleği VRAM analizi, Roofline modeli (Bellek-Bant-Genişliği vs. Hesaplama Darboğazı) ve INT4/NF4 hızlanma faktörleri.

### Adım 1: Kütüphanelerin Yüklenmesi ve Ortam Yapılandırması

In [ ]:
import math
import json
from pathlib import Path
import torch
import torch.nn as nn
import torch.nn.functional as F

from day29.mini_project.src.custom_nlp_engine import (
    MerinosBPETokenizer,
    RotaryEmbedding,
    RMSNorm,
    SwiGLUFFN,
    MoEFeedForward,
    CausalSelfAttention,
    TransformerBlock,
    MerinosCausalLM
)
from day29.mini_project.src.functional_backend import FunctionalEquivalenceEngine
from day29.mini_project.src.hardware_profiler import HardwareProfiler
from day29.mini_project.src.models import NanoLLMConfig
from day29.mini_project.src.visualizer import generate_diagnostic_panel

print(f"PyTorch Sürümü: {torch.__version__}")
print(f"CUDA Erişilebilirliği: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Aktif GPU: {torch.cuda.get_device_name(0)}")

### Adım 2: Halı Dokuma Alanına Özel Altkelime Tokenizer'ı (`MerinosBPETokenizer`)

In [ ]:
tokenizer = MerinosBPETokenizer(vocab_size=1024)
sample_text = "[ATKI] kopma sensoru alarm verdi dokuma tezgahi durduruldu [ARIZA]"

tokens = tokenizer.encode(sample_text, add_special_tokens=True)
decoded = tokenizer.decode(tokens, skip_special_tokens=False)

print(f"Orijinal Metin: {sample_text}")
print(f"Token Dizisi (Uzunluk: {len(tokens)}): {tokens}")
print(f"Geri Çözümlenen Metin: {decoded}")
print(f"Özel Belirteç Kimlikleri -> BOS: {tokenizer.bos_id}, EOS: {tokenizer.eos_id}, [ATKI]: {tokenizer.token_to_id['[ATKI]']}")

### Adım 3: Döner Konum Gömme (RoPE — Rotary Positional Embedding) Mekaniği
RoPE, pozisyonu mutlak bir vektör eklemek yerine Query ve Key vektörlerini 2 boyutlu alt-uzaylarda karmaşık sayılar gibi döndürerek kodlar:
$$\mathbf{R}_{\Theta, m}^d \mathbf{x}_m = \begin{pmatrix} \cos m\theta_1 & -\sin m\theta_1 \\ \sin m\theta_1 & \cos m\theta_1 \end{pmatrix} \begin{pmatrix} x_m^{(1)} \\ x_m^{(2)} \end{pmatrix}$$

In [ ]:
rope = RotaryEmbedding(dim=32, max_position_embeddings=128)
q = torch.randn(1, 4, 8, 32)  # (Batch, Heads, SeqLen, HeadDim)
k = torch.randn(1, 4, 8, 32)

q_rot, k_rot = rope(q, k, seq_len=8)
print(f"Q Tensörü Şekli: {q.shape} -> RoPE Sonrası: {q_rot.shape}")
print(f"İlk Token ile Son Token Arasındaki Kosinüs Benzerliği: {F.cosine_similarity(q_rot[0, 0, 0], q_rot[0, 0, 7], dim=-1).item():.4f}")

### Adım 4: RMSNorm & SwiGLU Gated Feed-Forward Ağı

In [ ]:
norm = RMSNorm(hidden_size=64)
x = torch.randn(2, 6, 64) * 3.5
normed_x = norm(x)

swiglu = SwiGLUFFN(hidden_size=64, intermediate_size=128)
ffn_out = swiglu(normed_x)

print(f"RMSNorm Çıkış Şekli: {normed_x.shape}, RMS Değeri: {torch.sqrt(torch.mean(normed_x**2, dim=-1))[0, 0].item():.4f}")
print(f"SwiGLU Çıkış Şekli: {ffn_out.shape}")

### Adım 5: Causal Self-Attention & Grouped Query Attention (GQA) & KV Önbellekleme

In [ ]:
cfg = NanoLLMConfig(vocab_size=1024, d_model=96, n_heads=6, n_kv_heads=2, n_layers=2, intermediate_size=256, max_seq_len=128)
attn = CausalSelfAttention(cfg)

attn_input = torch.randn(1, 10, cfg.d_model)
attn_output, present_kv = attn(attn_input)

print(f"GQA Dikkat Çıkışı: {attn_output.shape}")
print(f"Sorgu Başlık Sayısı: {cfg.n_heads}, KV Başlık Sayısı: {cfg.n_kv_heads} (Oran: {cfg.n_heads // cfg.n_kv_heads}:1)")
print(f"Önbellek Anahtar Şekli: {present_kv[0].shape}, Değer Şekli: {present_kv[1].shape}")

### Adım 6: Özel Merinos Nano-LLM Motoru ve Otoregresif Üretim Döngüsü

In [ ]:
nano_model = MerinosCausalLM(cfg)
param_info = nano_model.count_parameters()
print(f"Toplam Parametre Sayısı: {param_info['total']:,} ({param_info['total'] / 1e6:.2f}M)")

prompt_text = "[ATKI] kopma"
p_tokens = tokenizer.encode(prompt_text, add_special_tokens=False)
gen_tokens = nano_model.generate(p_tokens, max_new_tokens=15, temperature=0.7, top_k=20, use_cache=True)
print(f"Girdi Belirteçleri: {p_tokens}")
print(f"Üretilen Belirteçler ({len(gen_tokens)} adet): {gen_tokens}")
print(f"Çözümlenen Metin: {tokenizer.decode(gen_tokens, skip_special_tokens=False)}")

### Adım 6.1: Yeni Nesil Mimari: Sparse Mixture of Experts (MoE) & DeepSeek-Tarzı Shared Expert

In [ ]:
moe_cfg = NanoLLMConfig(
    vocab_size=1024, d_model=96, n_heads=4, n_kv_heads=2, n_layers=2, intermediate_size=256, max_seq_len=128,
    use_moe=True, num_experts=4, num_experts_per_tok=2, use_shared_expert=True, attention_logit_soft_capping=30.0
)
moe_model = MerinosCausalLM(moe_cfg)
moe_params = moe_model.count_parameters()
print(f"[MoE] Toplam Parametre: {moe_params['total']:,} ({moe_params['total']/1e6:.2f}M)")
print(f"[MoE] Token Başına Aktif Parametre: {moe_params['active_per_token']:,} ({moe_params['active_per_token']/1e6:.2f}M)")
print(f"Hesaplama Tasarrufu: %{(1.0 - moe_params['active_per_token'] / moe_params['total']) * 100:.1f}")

moe_gen = moe_model.generate(p_tokens, max_new_tokens=12, temperature=0.0, use_cache=True)
print(f"MoE Çıktısı: {tokenizer.decode(moe_gen, skip_special_tokens=False)}")

### Adım 7: PyTorch vs. TensorFlow Fonksiyonel Eşdeğerlik Kataloğu

In [ ]:
catalog = FunctionalEquivalenceEngine.get_catalog()
print(f"Katalogdaki Eşdeğerlik Sayısı: {len(catalog)}\n")
for item in catalog[:4]:
    print(f"* [{item.category}] {item.operation_name}")
    print(f"  - PyTorch   : {item.pytorch_syntax}")
    print(f"  - TensorFlow: {item.tensorflow_syntax}")
    print(f"  - Mekanizma : {item.description}\n")

### Adım 8: Donanım VRAM ve KV Önbellek Profillemesi

In [ ]:
profiler = HardwareProfiler()
candidate = profiler.candidate_models[0]  # Qwen 2.5 0.5B

profiles = profiler.profile_model_quantization(candidate, hardware_id="rtx_4060_8gb")
print(f"Model: {candidate.model_name} ({candidate.params_billion}B parametre)")
for p in profiles:
    print(f"Hassasiyet: {p.precision_type:<4} | Ağırlık: {p.model_size_gb:>5.2f} GB | KV/1K: {p.kv_cache_per_1k_tokens_mb:>5.2f} MB | Sıkıştırma: {p.compression_ratio:>4.1f}x | Hızlanma: {p.estimated_speedup:>4.1f}x")

### Adım 9: Donanım Roofline Modeli ve Darboğaz Analizi

In [ ]:
point_decode = profiler.analyze_roofline("rtx_4060_8gb", operational_intensity=1.5)
point_train = profiler.analyze_roofline("rtx_4060_8gb", operational_intensity=1200.0)

print(f"[Decoding (Token Üretimi)]: Yoğunluk: {point_decode.arithmetic_intensity} FLOP/B -> Rejim: {point_decode.operational_regime} (Erişilebilir: {point_decode.achievable_tflops} TFLOPS)")
print(f"[Training (LoRA İnce Ayar)]: Yoğunluk: {point_train.arithmetic_intensity} FLOP/B -> Rejim: {point_train.operational_regime} (Erişilebilir: {point_train.achievable_tflops} TFLOPS)")

### Adım 10: SLM Kıyaslama Raporu ve 300 DPI Master Teşhis Paneli

In [ ]:
benchmark = profiler.benchmark_all_candidates(hardware_id="rtx_4060_8gb", preferred_precision="INT4")
print(f"Seçilen En Uygun Minimal SLM: {benchmark['selected_slm'].upper()}")

output_panel = Path("outputs/llm_profiling_diagnostic_panel.png")
generate_diagnostic_panel(profiler, output_path=output_panel, dpi=300)
print(f"Master Teşhis Paneli Başarıyla Kaydedildi: {output_panel}")